# 05 — Perspective Transformation and Planar Homography

**Objective:** Correct perspective distortion by mapping four selected points from an image onto a rectangle using a planar homography H.

## Conceptual distinction

```
Lens distortion  →  corrected with  K, D  (Notebook 04)
Perspective warp →  corrected with  H     (this notebook)
```

Perspective distortion makes flat objects (a floor tile, a document, a road marking) appear trapezoidal. We flatten it using a **homography**: a 3×3 projective transform with 8 degrees of freedom, uniquely determined by 4 point correspondences.

**Order of operations:** Always apply lens correction **first**, then perspective rectification.


In [ ]:
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

SRC = Path('..') / 'src'
if str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

from calibvision.dewarper import PerspectiveRectifier
from calibvision.profile import CalibrationProfile
from calibvision.dewarper import LensDewarper

PROJECT_ROOT = Path('..').resolve()
CALIB_DIR    = PROJECT_ROOT / 'data' / 'calibration_images'
PROFILE_PATH = PROJECT_ROOT / 'configs' / 'saved_profiles' / 'synthetic_demo.yaml'
print('Setup complete')

In [ ]:
# ── 5.1  Synthetic demo: warp a rectangle into a trapezoid, then recover it ──
#
# We create a known rectangle (e.g. a flat document), apply a known perspective
# warp to simulate a camera angle, then recover the rectangle using homography.
# This lets us measure recovery accuracy.

def make_flat_document(w: int = 600, h: int = 400) -> np.ndarray:
    """Simulate a flat document with visible content for easy visual inspection."""
    doc = np.full((h, w, 3), 245, dtype=np.uint8)
    # Border
    cv2.rectangle(doc, (10, 10), (w-10, h-10), (30, 30, 30), 3)
    # Simulated text lines
    for i, y in enumerate(range(60, h-40, 40)):
        cv2.rectangle(doc, (30, y), (w-30, y+20), (100, 100, 200), -1)
    # Title block
    cv2.rectangle(doc, (30, 15), (w-30, 50), (60, 120, 200), -1)
    cv2.putText(doc, 'DOCUMENT RECTIFICATION TEST',
                (40, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    # Corner markers
    for corner in [(15,15), (w-25,15), (w-25,h-25), (15,h-25)]:
        cv2.circle(doc, corner, 8, (0, 200, 100), -1)
    return doc


DOC_W, DOC_H = 600, 400
flat_doc = make_flat_document(DOC_W, DOC_H)

# Apply a known perspective warp — simulates looking at the doc from an angle
canvas_w, canvas_h = 800, 600
canvas = np.full((canvas_h, canvas_w, 3), 180, dtype=np.uint8)

# Source corners of the flat doc (TL, TR, BR, BL)
src_corners = np.float32([[0,0], [DOC_W,0], [DOC_W,DOC_H], [0,DOC_H]])
# Destination: trapezoid (simulates perspective)
dst_corners = np.float32([[100,50], [680,80], [720,530], [60,550]])

H_fwd = cv2.getPerspectiveTransform(src_corners, dst_corners)
warped_into_canvas = cv2.warpPerspective(flat_doc, H_fwd, (canvas_w, canvas_h))
mask = cv2.warpPerspective(
    np.ones((DOC_H, DOC_W), dtype=np.uint8) * 255, H_fwd, (canvas_w, canvas_h)
)
canvas[mask > 0] = warped_into_canvas[mask > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(flat_doc, cv2.COLOR_BGR2RGB))
axes[0].set_title('Ground truth: flat document', fontsize=9)
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
# Draw the selected corner points
for pt in dst_corners:
    axes[1].plot(pt[0], pt[1], 'ro', markersize=8)
axes[1].set_title('Perspective-distorted (trapezoid)\nRed dots = corners to select', fontsize=9)
axes[1].axis('off')
plt.suptitle('Step 1: Simulated perspective distortion', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.2  Recover the rectangle using PerspectiveRectifier ────────────────────
rectifier = PerspectiveRectifier(output_width=DOC_W, output_height=DOC_H)

# The four corner points we 'clicked' on the distorted image (known in this demo)
# In real use, these come from user clicks or automatic detection
src_pts = dst_corners.copy()   # shape (4, 2) — TL, TR, BR, BL
recovered = rectifier.rectify(canvas, src_pts)

# Measure pixel accuracy vs ground truth
diff = cv2.absdiff(
    cv2.cvtColor(flat_doc, cv2.COLOR_BGR2GRAY),
    cv2.cvtColor(recovered, cv2.COLOR_BGR2GRAY)
)
mae = diff.mean()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
axes[0].set_title('Input (distorted)')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(recovered, cv2.COLOR_BGR2RGB))
axes[1].set_title('Rectified output')
axes[1].axis('off')

axes[2].imshow(diff, cmap='hot')
axes[2].set_title(f'|rectified − ground truth|\nMAE = {mae:.2f} px')
axes[2].axis('off')

plt.suptitle('Perspective rectification using planar homography', fontsize=11)
plt.tight_layout()
plt.show()

print(f'Mean absolute pixel error vs ground truth: {mae:.2f}')
print(f'(Small error expected due to bilinear interpolation rounding)')

In [ ]:
# ── 5.3  order_points helper demo ────────────────────────────────────────────
# In real use, users click four corners in arbitrary order.
# order_points() sorts them consistently regardless of click order.

arbitrary_order = np.array([
    [720, 530],   # BR
    [100, 50],    # TL  
    [60,  550],   # BL
    [680, 80],    # TR
], dtype=np.float32)

ordered = rectifier.order_points(arbitrary_order)
labels = ['TL', 'TR', 'BR', 'BL']
print('Input points (arbitrary order):')
for pt in arbitrary_order:
    print(f'  {pt}')
print('\nOrdered [TL, TR, BR, BL]:')
for label, pt in zip(labels, ordered):
    print(f'  {label}: {pt}')

result_ordered = rectifier.rectify_from_clicks(canvas, arbitrary_order)
result_manual  = rectifier.rectify(canvas, src_pts)
diff_check = cv2.absdiff(result_ordered, result_manual)
print(f'\nMax difference between ordered and manual: {diff_check.max()}')
assert diff_check.max() < 2, 'order_points output does not match manual ordering!'
print('✓ order_points produces consistent results')

In [ ]:
# ── 5.4  Full pipeline: lens correction → perspective rectification ────────────
# Apply both corrections in the correct sequence on a calibration image.

profile  = CalibrationProfile.load(PROFILE_PATH)
dewarper = LensDewarper(profile, alpha=0.5)

sample_path = sorted(CALIB_DIR.glob('*.png'))[0]
raw = cv2.imread(str(sample_path))

# Step 1: lens correction
undistorted = dewarper.undistort(raw)

# Step 2: perspective rectification on the entire image (square selection)
h, w = undistorted.shape[:2]
margin = 30
full_pts = np.array([
    [margin, margin],
    [w - margin, margin],
    [w - margin, h - margin],
    [margin, h - margin]
], dtype=np.float32)
rect_full = PerspectiveRectifier(output_width=w, output_height=h)
rectified = rect_full.rectify(undistorted, full_pts)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, img, title in zip(axes,
    [raw, undistorted, rectified],
    ['1. Raw image', '2. Lens undistorted', '3. Perspective rectified']):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=9)
    ax.axis('off')

plt.suptitle('Full pipeline: lens correction → perspective rectification', fontsize=11)
plt.tight_layout()
plt.show()

## Summary

| Step | Implemented |
|---|---|
| Homography estimation (4 points) | ✓ `cv2.getPerspectiveTransform` |
| Perspective warp | ✓ `cv2.warpPerspective` |
| Arbitrary point ordering | ✓ `order_points()` |
| Full pipeline (lens + perspective) | ✓ |
| Recovery accuracy measured | ✓ MAE vs ground truth |

## Limitation

Four-point homography is exact only when the selected region is **planar**. Applying it to non-planar scenes (curved surfaces, 3-D objects) will introduce warping artefacts.

## Next step

Open `06_calibration_quality_assessment.ipynb` for sensitivity analysis and reprojection error deep-dive.
